In [4]:

import pandas as pd
import numpy as np


In [5]:
#Load raw dataset 

df_raw = pd.read_csv('../data/raw/WVS_Survey_Trimmed.csv', low_memory=False)


In [4]:
need = ['B_COUNTRY_ALPHA', 'W_WEIGHT', 'Q51', 'Q53', 'Q54', 'Q55']
print([c for c in need if c not in df_raw.columns])   # should print []

[]


In [6]:
need = ['B_COUNTRY_ALPHA', 'W_WEIGHT', 'Q51', 'Q53', 'Q54', 'Q55']
df = df_raw[need].copy()

# make sure the survey items and weight are numeric
for c in ['W_WEIGHT', 'Q51', 'Q53', 'Q54', 'Q55']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# WVS missing codes (all negative) -> NaN
items = {'Q51': 'food', 'Q53': 'medicine', 'Q54': 'cash_income', 'Q55': 'shelter'}
for q in items:
    df[q] = df[q].where(df[q] > 0)

country_map = {
    'MMR': 'Myanmar', 'IDN': 'Indonesia', 'MYS': 'Malaysia',
    'PHL': 'Philippines', 'SGP': 'Singapore', 'VNM': 'Vietnam', 'THA': 'Thailand'
}
sea = df[df['B_COUNTRY_ALPHA'].isin(country_map)].copy()

# 1=Often, 2=Sometimes, 3=Rarely, 4=Never -> 3,2,1,0
for q in items:
    sea[q] = 4 - sea[q]

In [7]:
print(sea[list(items)].apply(lambda s: s.value_counts(dropna=False).sort_index()))

      Q51   Q53   Q54   Q55
0.0  7690  6808  6182  9611
1.0  1703  2380  2211  1035
2.0  1640  1745  2415   633
3.0   585   677   806   332
NaN     7    15    11    14


In [9]:
def wmean(g, col):
    g = g.dropna(subset=[col, 'W_WEIGHT'])
    return np.average(g[col], weights=g['W_WEIGHT'])

res = pd.DataFrame({
    name: {c: wmean(g, q) for c, g in sea.groupby('B_COUNTRY_ALPHA')}
    for q, name in items.items()
})
res.index = res.index.map(country_map)
res.round(3)

,food,medicine,cash_income,shelter
Indonesia,0.886,1.115,0.968,0.308
Myanmar,0.346,0.282,0.882,0.231
Malaysia,0.460,0.617,0.788,0.313
Philippines,0.893,0.990,1.129,0.388
Singapore,0.128,0.155,0.301,0.064
Thailand,0.811,0.780,1.020,0.613
Vietnam,0.214,0.369,0.612,0.068


Ethical Values and Norms Q177 - 181

In [1]:
# Q177-181: justifiability of economic norm violations (1 = Never justifiable, 10 = Always justifiable)
questions = {
    'Q177': 'benefits',         # Claiming government benefits you're not entitled to
    'Q178': 'fare_avoidance',   # Avoiding a fare on public transport
    'Q179': 'stealing',         # Stealing property
    'Q180': 'tax_cheating',     # Cheating on taxes if you have a chance
    'Q181': 'bribery'           # Someone accepting a bribe in the course of their duties
}

In [2]:
sea_countries = {
    'IDN': 'Indonesia',
    'MMR': 'Myanmar',
    'MYS': 'Malaysia',
    'PHL': 'Philippines',
    'SGP': 'Singapore',
    'THA': 'Thailand',
    'VNM': 'Vietnam'
}


In [6]:
df = df_raw[df_raw['B_COUNTRY_ALPHA'].isin(sea_countries.keys())].copy()

In [7]:
def weighted_mean(group, col):
    valid = group[group[col] > 0]   # drop negative missing/don't-know codes
    return np.average(valid[col], weights=valid['W_WEIGHT'])

In [8]:
results = {}
for q_col, label in questions.items():
    results[label] = df.groupby('B_COUNTRY_ALPHA').apply(lambda g: weighted_mean(g, q_col))

In [9]:
result_df = pd.DataFrame(results)
result_df.index = result_df.index.map(sea_countries)
result_df = result_df.round(3)
print(result_df)

                 benefits  fare_avoidance  stealing  tax_cheating  bribery
B_COUNTRY_ALPHA                                                           
Indonesia           3.343           2.802     1.583         2.582    1.939
Myanmar             3.655           3.161     1.489         1.622    1.722
Malaysia            4.395           3.979     3.100         3.409    3.085
Philippines         5.211           4.805     3.646         4.041    4.104
Singapore           2.534           1.823     1.351         1.538    1.412
Thailand            2.223           2.231     1.832         1.744    1.774
Vietnam             4.524           3.424     2.495         2.832    2.898


Rescale to a 0 to 1 scale so that we can compare fairly, since these are measured on a 1 to 10 scale and the earlier questions are measured on a 1 to 4 scale.

In [10]:
result_df_rescaled = ((result_df - 1) / 9).round(3)
print(result_df_rescaled)

                 benefits  fare_avoidance  stealing  tax_cheating  bribery
B_COUNTRY_ALPHA                                                           
Indonesia           0.260           0.200     0.065         0.176    0.104
Myanmar             0.295           0.240     0.054         0.069    0.080
Malaysia            0.377           0.331     0.233         0.268    0.232
Philippines         0.468           0.423     0.294         0.338    0.345
Singapore           0.170           0.091     0.039         0.060    0.046
Thailand            0.136           0.137     0.092         0.083    0.086
Vietnam             0.392           0.269     0.166         0.204    0.211


Personal norms 

In [43]:
pd.set_option("display.max_columns", None)

print(df_raw.shape)
print(df_raw.columns.tolist())
print(df_raw.head(10))

(11625, 260)
['version', 'doi', 'A_WAVE', 'A_YEAR', 'A_STUDY', 'B_COUNTRY', 'B_COUNTRY_ALPHA', 'C_COW_NUM', 'C_COW_ALPHA', 'D_INTERVIEW', 'S007', 'J_INTDATE', 'FW_START', 'FW_END', 'K_TIME_START', 'K_TIME_END', 'K_DURATION', 'Q_MODE', 'N_REGION_ISO', 'N_REGION_WVS', 'N_REGION_NUTS2', 'N_REG_NUTS1', 'N_TOWN', 'G_TOWNSIZE', 'G_TOWNSIZE2', 'H_SETTLEMENT', 'H_URBRURAL', 'I_PSU', 'O1_LONGITUDE', 'O2_LATITUDE', 'L_INTERVIEWER_NUMBER', 'S_INTLANGUAGE', 'LNGE_ISO', 'E_RESPINT', 'F_INTPRIVACY', 'E1_LITERACY', 'W_WEIGHT', 'S018', 'PWGHT', 'S025', 'Q51', 'Q53', 'Q54', 'Q55', 'Q177', 'Q178', 'Q179', 'Q180', 'Q181', 'Q182', 'Q183', 'Q184', 'Q185', 'Q186', 'Q187', 'Q188', 'Q189', 'Q190', 'Q191', 'Q192', 'Q193', 'Q194', 'Q195', 'Q287', 'Q288', 'Q288R', 'Y001', 'Y002', 'Y003', 'SACSECVAL', 'RESEMAVAL', 'I_AUTHORITY', 'I_NATIONALISM', 'I_DEVOUT', 'DEFIANCE', 'I_RELIGIMP', 'I_RELIGBEL', 'I_RELIGPRAC', 'DISBELIEF', 'I_NORM1', 'I_NORM2', 'I_NORM3', 'RELATIVISM', 'I_TRUSTARMY', 'I_TRUSTPOLICE', 'I_TRUSTCOU

In [57]:
# only focus on the ASEAN countries

country_col = "B_COUNTRY_ALPHA"

country_map = {
    'MMR': 'Myanmar', 'IDN': 'Indonesia', 'MYS': 'Malaysia',
    'PHL': 'Philippines', 'SGP': 'Singapore', 'VNM': 'Vietnam', 'THA': 'Thailand'
}

df_sea = df_raw[df_raw[country_col].isin(country_map.keys())].copy()



In [58]:
# Use the Q182-Q195 columns only
q_cols = [f"Q{i}" for i in range(182, 196)]

In [59]:
#change the variable into numeric and remove missing code
df_sea[q_cols] = df_sea[q_cols].apply(pd.to_numeric, errors="coerce")
df_sea[q_cols] = df_sea[q_cols].where(df_sea[q_cols] > 0)

In [ ]:
question_means = df_sea.groupby(country_col)[q_cols].mean()
print(question_means.round(2))

                 Q182  Q183  Q184  Q185  Q186  Q187  Q188  Q189  Q190  Q191  \
B_COUNTRY_ALPHA                                                               
IDN              1.61  1.57  1.58  2.69  1.64  1.48  2.19  1.67  2.44  1.86   
MMR              1.72  1.85  1.41  2.45  1.63  1.36  1.95  1.53  5.04  1.44   
MYS              3.59  3.39  3.52  4.64  3.82  3.15  3.95  3.13  4.47  3.28   
PHL              4.57  3.77  3.02  4.20  4.31  3.13  3.62  3.44  3.96  3.73   
SGP              3.45  2.78  3.29  4.21  4.04  2.46  4.19  1.32  3.36  1.58   
THA              4.21  2.49  2.39  4.58  4.48  2.49  2.76  2.00  2.32  1.90   
VNM              5.64  3.95  4.17  5.23  5.53  4.47  4.88  2.80  4.54  3.05   

                 Q192  Q193  Q194  Q195  
B_COUNTRY_ALPHA                          
IDN              1.73  1.52  1.78  3.38  
MMR              1.55  1.39  1.43  3.93  
MYS              3.10  3.59  3.15  4.58  
PHL              4.18  3.87  3.99  4.13  
SGP              1.30  2.87  1.55  4

In [62]:
#Average the 14 question means for each country
question_means["mean personal violation score"] = question_means.mean(axis=1)

In [64]:
#puting the average personal violation score inot a table
result = question_means[["mean personal violation score"]].reset_index()
result["Country"] = result[country_col].map(country_map)
result = (
    result[["Country", "mean personal violation score"]]
    .sort_values("mean personal violation score", ascending=False)
    .reset_index(drop=True)
)
result["mean personal violation score"] = result["mean personal violation score"].round(2)

print(result)
result.to_csv("country_mean_personal_violation_score.csv", index=False)

       Country  mean personal violation score
0      Vietnam                           4.27
1  Philippines                           3.85
2     Malaysia                           3.67
3    Singapore                           2.94
4     Thailand                           2.91
5      Myanmar                           2.05
6    Indonesia                           1.94
